# 1. Setup & Environment Initialization

Handles imports, checks GPU acceleration, and points to dataset configuration


In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import torch

# Configuration Parameters
DATA_YAML = Path("../data/CHV_yolo/data.yaml").resolve()
EPOCHS_PROTOTYPE = 30
EPOCHS_PRODUCTION = 80
IMGSZ = 640
BATCH = 16
WORKERS = 4

# Bypassing Windows OpenMP conflict if running locally
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

assert DATA_YAML.exists(), f"data.yaml not found at {DATA_YAML}. Check your path!"

print(f"Using dataset config: {DATA_YAML}")
print("--- Hardware Check ---")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU Acceleration Active: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Grid search will be exceptionally slow.")

Using dataset config: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\data\CHV_yolo\data.yaml
--- Hardware Check ---
GPU Acceleration Active: NVIDIA GeForce RTX 5060 Ti


# 2. Candidates Roster

Short-Run Candidate Screening Loop


In [2]:
# Roster of candidate models to evaluate
candidates = {
    "YOLOv8_Nano": "yolov8n.pt",
    "YOLOv9_Tiny": "yolov9t.pt",
    "YOLOv10_Nano": "yolov10n.pt",
    "YOLO11_Nano": "yolo11n.pt",
    "YOLO12_Nano": "yolo12n.pt",
    "YOLO26_Nano": "yolo26n.pt",
}

results_log = []
print(f"Configured {len(candidates)} architectures for screening.")

Configured 6 architectures for screening.


# 3. Short-Run Candidate Screening Loop

Each model is benchmarked for 30 epochs using standard native augmentations.


In [3]:
print("🚀 Starting Phase 1: Candidate Screening (30 Epochs each)...")

for name, weight in candidates.items():
    print(
        f"\n========================================\nTraining Candidate: {name}\n========================================"
    )
    try:
        model = YOLO(weight)

        # Moderate native augmentations for a clean prototype baseline
        model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS_PROTOTYPE,
            imgsz=IMGSZ,
            batch=BATCH,
            workers=WORKERS,
            device=DEVICE,
            project="PPE_Prototype",
            name=f"{name}_baseline",
            mosaic=1.0,
            scale=0.5,  # Mild scaling
            hsv_v=0.4,  # Mild brightness variance
            translate=0.1,  # Mild shifting
            plots=False,
        )

        # Immediate Validation
        print(f"Evaluating metrics for {name}...")
        metrics = model.val(data=str(DATA_YAML), split="val")

        # Extract individual safety-critical class metrics safely
        try:
            recall_person = float(metrics.box.r[0])
            recall_helmet = float(metrics.box.r[1])
        except IndexError:
            print(
                "⚠️ Index error reading specific classes. Check your data.yaml 'names' mapping."
            )
            recall_person, recall_helmet = 0.0, 0.0

        map50 = float(metrics.box.map50)
        speed_ms = float(metrics.speed.get("inference", 0.0))

        results_log.append(
            {
                "Model": name,
                "mAP50": round(map50, 4),
                "Recall_Person": round(recall_person, 4),
                "Recall_Helmet": round(recall_helmet, 4),
                "Composite_Safety_Recall": round(
                    (recall_person + recall_helmet) / 2, 4
                ),
                "Inference_Speed_ms": round(speed_ms, 2),
                "Weights_Path": f"PPE_Prototype/{name}_baseline/weights/best.pt",
            }
        )

        # Explicit VRAM Cleanup (Prevents out-of-memory leaks across iterations)
        del model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ Failed to successfully evaluate {name}. Error: {e}")

print("\n✅ Phase 1 Screening Complete.")

🚀 Starting Phase 1: Candidate Screening (30 Epochs each)...

Training Candidate: YOLOv8_Nano
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\data\CHV_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=t

# 4. Triage Matrix & Visual Charts

This cell compiles the run statistics into a table and builds a scatter chart displaying the performance-versus-latency trade-off.


In [4]:
if not results_log:
    raise RuntimeError("No models finished training successfully. Check errors above.")

print("📊 Compiling Evaluation Matrix...")
df = pd.DataFrame(results_log)

# Priority Sorting Rule: Safety Recall -> Overall Map -> Latency Speed
df = df.sort_values(
    by=["Composite_Safety_Recall", "mAP50", "Inference_Speed_ms"],
    ascending=[False, False, True],
).reset_index(drop=True)

# Save out evaluation artifacts
df.to_csv("prototype_candidates_results.csv", index=False)

print("\n--- Final Matrix Results (Sorted by Safety-First) ---")
try:
    from IPython.display import display

    display(df)
except ImportError:
    print(df.to_string())

# Plotting the trade-offs
plt.figure(figsize=(10, 6))
plt.scatter(
    df["Inference_Speed_ms"],
    df["mAP50"],
    s=df["Composite_Safety_Recall"]
    * 1200,  # Bubble size represents safety recall strength
    alpha=0.6,
    edgecolors="w",
    color="darkcyan",
)

for i, row in df.iterrows():
    plt.text(
        row["Inference_Speed_ms"] + 0.04,
        row["mAP50"],
        row["Model"],
        fontsize=10,
        weight="bold",
    )

plt.xlabel("Inference Speed (ms/image) — Lower is Faster")
plt.ylabel("mAP50 Accuracy — Higher is Better")
plt.title(
    "Prototype Trade-offs: Speed vs. mAP50 vs. Composite Safety Recall (Bubble Size)"
)
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("architecture_tradeoffs.png", bbox_inches="tight")
plt.show()

📊 Compiling Evaluation Matrix...

--- Final Matrix Results (Sorted by Safety-First) ---


,Model,mAP50,Recall_Person,Recall_Helmet,Composite_Safety_Recall,Inference_Speed_ms,Weights_Path
0,YOLO11_Nano,0.8988,0.8992,0.8778,0.8885,2.16,PPE_Prototype/YOLO11_Nano_baseline/weights/bes...
1,YOLOv8_Nano,0.9035,0.8911,0.8691,0.8801,2.93,PPE_Prototype/YOLOv8_Nano_baseline/weights/bes...
2,YOLOv9_Tiny,0.9053,0.8760,0.8732,0.8746,3.88,PPE_Prototype/YOLOv9_Tiny_baseline/weights/bes...
3,YOLO12_Nano,0.9095,0.8786,0.8587,0.8686,3.01,PPE_Prototype/YOLO12_Nano_baseline/weights/bes...
4,YOLOv10_Nano,0.8886,0.8553,0.8755,0.8654,2.66,PPE_Prototype/YOLOv10_Nano_baseline/weights/be...
5,YOLO26_Nano,0.8821,0.8475,0.8192,0.8334,2.42,PPE_Prototype/YOLO26_Nano_baseline/weights/bes...


<Figure size 1000x600 with 1 Axes>

# 5. Production Deep Tuning & Weights Extraction

Extracts the highest scoring prototype from the table, targets its weights, runs a full training routine, and compiles the model architecture into an optimized format


In [5]:
# %%
import os
from pathlib import Path

winner = df.iloc[0]
winning_model_name = winner["Model"]
print(f"🏆 The Chosen Architecture is: {winning_model_name}")

# 1. Dynamically search the current relative workspace for the winning weights
current_dir = Path.cwd()
# Search all subdirectories for a path containing the model name and ending in "weights/best.pt"
found_weights = list(current_dir.rglob(f"*{winning_model_name}*/weights/best.pt"))

if not found_weights:
    raise FileNotFoundError(
        f"❌ Could not find weights for {winning_model_name} in any subfolder."
    )

# 2. If multiple runs exist, dynamically grab the most recently created/modified one
dynamic_weights_path = max(found_weights, key=os.path.getmtime)

# Convert to a clean relative path for display
relative_path = dynamic_weights_path.relative_to(current_dir)
print(f"🔍 Dynamically located latest checkpoints at:\n👉 {relative_path}\n")

# 3. Re-initialize clean instance pointing to the dynamic path
production_model = YOLO(str(dynamic_weights_path))

print(f"🔥 Initiating full production schedule ({EPOCHS_PRODUCTION} Epochs)...")
production_model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_PRODUCTION,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    project="PPE_Production",  # Creates a relative folder in your current directory
    name=f"{winning_model_name}_Final",
    plots=True,
)

# 4. Dynamically construct the final output path
final_pt_path = (
    Path("PPE_Production") / f"{winning_model_name}_Final" / "weights" / "best.pt"
)

print("\n========================================================")
print("🚀 PRODUCTION DEEP TUNING COMPLETE")
print("========================================================")
print(f"Your final deployment-ready PyTorch model is saved at:\n👉 {final_pt_path}")

🏆 The Chosen Architecture is: YOLO11_Nano
🔍 Dynamically located latest checkpoints at:
👉 runs\detect\PPE_Prototype\YOLO11_Nano_baseline\weights\best.pt

🔥 Initiating full production schedule (80 Epochs)...
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\data\CHV_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, 

## 6. Inference


In [ ]:
# %%
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print("🔍 Initiating Live Visual Verification on Test Split...")

# 1. Verification of the production model path
# If running this out of order, ensure final_pt_path points to your best.pt
if "final_pt_path" not in locals() or not final_pt_path.exists():
    # Dynamic fallback check if cell state was lost
    current_dir = Path.cwd()
    found_prod_weights = list(current_dir.rglob("PPE_Production/*/weights/best.pt"))
    if found_prod_weights:
        final_pt_path = max(found_prod_weights, key=os.path.getmtime)
    else:
        raise FileNotFoundError(
            "❌ Production model weights file ('best.pt') not found. Please run Cell 5 first!"
        )

print(r"Loading finalized production weights from:")
print(f"👉 {final_pt_path.resolve()}\n")
visual_model = YOLO(str(final_pt_path))

# 2. Automatically locate the holdout test image directory
test_images_dir = DATA_YAML.parent / "images" / "test"

# Fallback in case directory structure differs
if not test_images_dir.exists():
    # Broad scan fallback for safety
    print("⚠️ 'images/test' folder not found at default location. Scanning workspace...")
    found_test_dirs = list(DATA_YAML.parent.rglob("test"))
    if found_test_dirs:
        test_images_dir = found_test_dirs[0]
    else:
        raise FileNotFoundError(
            f"❌ Could not find a test image split folder relative to {DATA_YAML.parent}"
        )

# 3. Gather all image extensions
valid_extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
all_test_images = []
for ext in valid_extensions:
    all_test_images.extend(list(test_images_dir.glob(ext)))

if len(all_test_images) == 0:
    raise FileNotFoundError(
        f"❌ No images found inside the test directory: {test_images_dir}"
    )

# 4. Select exactly 9 random images (or maximum available if less than 9)
num_samples = min(9, len(all_test_images))
sampled_image_paths = random.sample(all_test_images, num_samples)
print(
    f"Successfully sampled {num_samples} random unseen images from the blind test split."
)

# 5. Execute batch inference
# Adjust conf (confidence threshold) here if you want to test strictness (e.g., 0.25 to 0.40)
inference_results = visual_model(
    sampled_image_paths, imgsz=IMGSZ, conf=0.25, verbose=False
)

# 6. Set up the 3x3 Matplotlib subplot canvas
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.ravel()  # Flatten the 2D grid array into 1D for easy indexing

print("🎨 Rendering bounding boxes onto the subplots...")
for idx, result in enumerate(inference_results):
    # .plot() returns a BGR numpy array containing the overlayed bounding boxes and labels
    annotated_bgr_img = result.plot()

    # Convert from OpenCV's BGR format to Matplotlib's expected RGB format
    annotated_rgb_img = cv2.cvtColor(annotated_bgr_img, cv2.COLOR_BGR2RGB)

    # Render onto the subplot grid
    axes[idx].imshow(annotated_rgb_img)
    axes[idx].set_title(f"Sample: {Path(result.path).name}", fontsize=10, weight="bold")
    axes[idx].axis("off")  # Clean layout with no distracting axis pixels

# Hide any leftover unused subplots if the folder contains fewer than 9 images
for empty_idx in range(num_samples, 9):
    axes[empty_idx].axis("off")

plt.tight_layout()
plt.suptitle(
    "🚨 Production Model Predictions: Unseen Holdout Test Set",
    fontsize=16,
    weight="bold",
    y=1.02,
)

# Save a static record of this inference layout to disk
plt.savefig("production_visual_verification.png", bbox_inches="tight", dpi=150)
plt.show()

print(
    "🏁 Visual Verification Phase Complete! Static output stored at: production_visual_verification.png"
)

🔍 Initiating Live Visual Verification on Test Split...
Loading finalized production weights from:
👉 C:\Github\smart-factory-safety-monitoring-system\ppe-detection\notebooks\runs\detect\PPE_Production\YOLO11_Nano_Final\weights\best.pt

Successfully sampled 9 random unseen images from the blind test split.
🎨 Rendering bounding boxes onto the subplots...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_53592\2647535027.py:81: UserWarning: Glyph 128680 (\N{POLICE CARS REVOLVING LIGHT}) missing from font(s) DejaVu Sans.
  plt.savefig("production_visual_verification.png", bbox_inches='tight', dpi=150)


<Figure size 1800x1800 with 9 Axes>

🏁 Visual Verification Phase Complete! Static output stored at: production_visual_verification.png
